In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
import casadi as ca
import pandas as pd

In [ ]:
# 1. set file path
file_path = "../Laptimeoptimization/Monza.csv"

# 2. read csv file
df = pd.read_csv(file_path)

# 3. look head
print(df.head())

# 4. rename the first row
##df.rename(columns={'# x_m': 'x_m'}, inplace=True)

# 5. read the columns
x = df['# x_m'].values
y = df['y_m'].values
w_left = df['w_tr_left_m'].values
w_right = df['w_tr_right_m'].values

In [ ]:
# Step 1: Compute arclengths
s = np.zeros(len(x))
s[1:] = np.cumsum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
s_max=s[-1] #the length of the track

# Step 2: Create cubic spline interpolators
x_spline = CubicSpline(s, x)
y_spline = CubicSpline(s, y)

# Step 3: Get the first and second derivative of the parameteriation
dx_ds = x_spline.derivative(1)
d2x_ds2 = x_spline.derivative(2)
dy_ds = y_spline.derivative(1)
d2y_ds2 = y_spline.derivative(2)

## Generate interpolated values for plotting
s_query = np.linspace(0, s_max, 500)
x_interp = x_spline(s_query)
y_interp = y_spline(s_query)

## Plot the spline
plt.figure(figsize=(8, 6))
plt.plot(x_interp, y_interp, label='Interpolated Track', color='blue')
##plt.plot(x, y, 'ro', label='Original Points')
plt.axis('equal')
plt.title('Race Track Centerline via Cubic Spline Interpolation')
plt.xlabel('x (meters)')
plt.ylabel('y (meters)')
plt.legend()
plt.grid(True)
plt.show()

# output parameterisation and curvature as a casadi instance
x_ref = ca.interpolant('x_ref', 'linear', [s_query], x_interp)
y_ref = ca.interpolant('y_ref', 'linear', [s_query], y_interp)
kappa_ref_ = dx_ds(s_query)*d2y_ds2(s_query)-dy_ds(s_query)*d2x_ds2(s_query)
kappa_ref = ca.interpolant('kappa', 'linear', [s_query], kappa_ref_)

In [ ]:
# ----------------------------------------------------------------------
# ============       PARAMETERS (easy to tune)        ==================
# ----------------------------------------------------------------------
ANG_MIN_DEG   = 3.0          # θ candidates below this are ignored      (°)
CURV_MIN      = 5e-4         # |κ| below this is considered "straight" (1/m)
SLIDING_WIN   = 20.0         # sliding-max window length along s (m)
DENSE_STEP    = 0.5          # dense sampling step for boundaries (m)
DELTA_S       = 300.0        # forward window for FOV (m)

# ----------------------------------------------------------------------
# ============  Helper functions: boundary, extrema, FOV  ==============
# ----------------------------------------------------------------------
from typing import Tuple

def build_boundaries(spline_x, spline_y,
                     w_left: np.ndarray, w_right: np.ndarray,
                     s_dense: np.ndarray
                     ) -> Tuple[
                         Tuple[np.ndarray, np.ndarray],   # left (x,y)
                         Tuple[np.ndarray, np.ndarray],   # right (x,y)
                         Tuple[np.ndarray, np.ndarray],   # normal (nx,ny)
                         np.ndarray                       # tangent norm
                     ]:
    """Return dense left/right boundary samples and local normals."""
    xc = spline_x(s_dense)
    yc = spline_y(s_dense)

    dx = spline_x.derivative(1)(s_dense)
    dy = spline_y.derivative(1)(s_dense)
    norm_t = np.hypot(dx, dy)
    tx, ty = dx / norm_t, dy / norm_t
    nx, ny = -ty, tx                                    # left normal

    w_left_d  = np.interp(s_dense, s, w_left)
    w_right_d = np.interp(s_dense, s, w_right)

    xl = xc + nx * w_left_d
    yl = yc + ny * w_left_d
    xr = xc - nx * w_right_d
    yr = yc - ny * w_right_d
    return (xl, yl), (xr, yr), (nx, ny), norm_t


def first_crossing(s0: float, p_ray: np.ndarray,
                   center_x: np.ndarray, center_y: np.ndarray,
                   s_dense: np.ndarray) -> float:
    """Return the first s on the centreline hit by the ray; ∞ if none."""
    # Get index of the segment where s0 lives
    i0 = np.searchsorted(s_dense, s0)
    P0 = np.array([center_x[i0], center_y[i0]])          # ray anchor

    for i in range(i0, len(center_x) - 1):
        A = np.array([center_x[i],     center_y[i]])
        B = np.array([center_x[i + 1], center_y[i + 1]])
        v = B - A
        M = np.column_stack((p_ray, -v))                 # [ray | -segment]
        rhs = A - P0
        try:
            t, u = np.linalg.solve(M, rhs)
        except np.linalg.LinAlgError:
            continue                                    # parallel lines
        if t >= 0 and 0.0 <= u <= 1.0:
            return s_dense[i] + u * (s_dense[i + 1] - s_dense[i])
    return np.inf


def has_local_extreme(theta: np.ndarray) -> bool:
    """Early-exit monotonicity test with 3 ° dead-zone."""
    falling = None
    for prev, curr in zip(theta[:-1], theta[1:]):
        if abs(curr) < np.deg2rad(ANG_MIN_DEG):          # angle too small
            continue
        if prev == curr:
            continue
        trend = curr > prev                              # rising?
        if falling is None:
            falling = trend
        elif trend != falling:                           # slope changed
            return True
        falling = trend
    return False


def compute_fov_all(spline_x, spline_y,
                    w_left: np.ndarray, w_right: np.ndarray,
                    s_query: np.ndarray,
                    delta_s: float = DELTA_S,
                    step: float = DENSE_STEP) -> np.ndarray:
    """Compute sfov(s) for every point in s_query."""
    # --- dense grid along the whole track --------------------------------
    s_dense = np.arange(0, s_max + step, step)
    (xl, yl), (xr, yr), (nx, ny), _ = build_boundaries(
        spline_x, spline_y, w_left, w_right, s_dense)
    center_x = spline_x(s_dense)
    center_y = spline_y(s_dense)

    sfov_raw = np.empty_like(s_query)

    for k, s0 in enumerate(s_query):
        idx0 = np.searchsorted(s_dense, s0)
        idx_w = np.searchsorted(s_dense, s0 + delta_s, side='right')

        # wrap-around for circular track
        if idx_w >= len(s_dense):
            extra = idx_w - len(s_dense) + 1
            s_ext       = np.concatenate((s_dense, s_dense[1:extra] + s_max))
            xl_ext      = np.concatenate((xl, xl[1:extra]))
            yl_ext      = np.concatenate((yl, yl[1:extra]))
            xr_ext      = np.concatenate((xr, xr[1:extra]))
            yr_ext      = np.concatenate((yr, yr[1:extra]))
            center_xext = np.concatenate((center_x, center_x[1:extra]))
            center_yext = np.concatenate((center_y, center_y[1:extra]))
        else:
            s_ext       = s_dense
            xl_ext, yl_ext = xl, yl
            xr_ext, yr_ext = xr, yr
            center_xext, center_yext = center_x, center_y

        # --- vehicle heading vector at s0 --------------------------------
        tx0 = spline_x.derivative(1)(s0)
        ty0 = spline_y.derivative(1)(s0)
        dir_vec = np.array([tx0, ty0]) / np.hypot(tx0, ty0)

        # --- helper to compute θ sequence --------------------------------
        def theta_arr(xb, yb):
            Vx = xb[idx0:idx_w] - spline_x(s0)
            Vy = yb[idx0:idx_w] - spline_y(s0)
            cross = dir_vec[0] * Vy - dir_vec[1] * Vx
            dot   = dir_vec[0] * Vx + dir_vec[1] * Vy
            theta = np.arctan2(cross, dot)
            # filter small |θ| here (ANG_MIN)
            mask  = np.abs(theta) >= np.deg2rad(ANG_MIN_DEG)
            return theta[mask], Vx[mask], Vy[mask]

        θL, VLx, VLy = theta_arr(xl_ext, yl_ext)
        θR, VRx, VRy = theta_arr(xr_ext, yr_ext)

        has_ext_L = has_local_extreme(θL)
        has_ext_R = has_local_extreme(θR)

        # --- curvature-based straight-road filter ------------------------
        kappa_now = (spline_x.derivative(1)(s0) * spline_y.derivative(2)(s0)
                     - spline_y.derivative(1)(s0) * spline_x.derivative(2)(s0))
        if abs(kappa_now) < CURV_MIN:
            # small curvature: treat as straight
            sfov_raw[k] = s0 + delta_s
            continue

        # if both sides miss extrema within δs, also treat as straight
        if not has_ext_L and not has_ext_R:
            sfov_raw[k] = s0 + delta_s
            continue

        # --- choose rays --------------------------------------------------
        s_inner = np.inf
        s_outer = np.inf

        if kappa_now < 0:  # right turn ⇒ right side is inner
            if has_ext_R:
                idx_R = np.argmin(θR)            # most negative
                p_ray = np.array([VRx[idx_R], VRy[idx_R]])
                p_ray /= np.linalg.norm(p_ray)
                s_inner = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
            if has_ext_L:
                idx_L = np.argmax(θL)            # largest positive
                p_ray = np.array([VLx[idx_L], VLy[idx_L]])
                p_ray /= np.linalg.norm(p_ray)
                s_outer = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
        else:            # left turn
            if has_ext_L:
                idx_L = np.argmax(θL)
                p_ray = np.array([VLx[idx_L], VLy[idx_L]])
                p_ray /= np.linalg.norm(p_ray)
                s_inner = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
            if has_ext_R:
                idx_R = np.argmin(θR)
                p_ray = np.array([VRx[idx_R], VRy[idx_R]])
                p_ray /= np.linalg.norm(p_ray)
                s_outer = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)

        sfov_raw[k] = min(s_inner, s_outer)

    # ------------------------------------------------------------------
    # Sliding-maximum filter: ensure sfov is non-decreasing
    # ------------------------------------------------------------------
    win = int(SLIDING_WIN / (s_query[1] - s_query[0]))  # window size in samples
    sfov_filt = np.copy(sfov_raw)
    running_max = sfov_raw[0]
    for i in range(len(sfov_raw)):
        running_max = max(running_max, sfov_raw[i])
        if i > win:
            running_max = max(sfov_raw[i - win:i + 1])
        sfov_filt[i] = running_max
    return sfov_filt

In [ ]:
sfov_vals = compute_fov_all(
    x_spline, y_spline, w_left, w_right,
    s_query, delta_s=DELTA_S)

sfov_ca = ca.interpolant('sfov', 'linear', [s_query], sfov_vals)

In [ ]:
# ---------- save_fov_to_csv.py ----------
import os
import pandas as pd

def save_sfov_csv(s_samples, sfov_samples,
                  in_csv_path: str,
                  out_name: str = "sfov_lookup.csv") -> str:
    """
    Save (s, sfov) pairs to a CSV file in the same folder as the input track.

    Parameters
    ----------
    s_samples     : 1-D array-like, arc-length samples.
    sfov_samples  : 1-D array-like, corresponding look-ahead values.
    in_csv_path   : str, path of the original track CSV (e.g. 'Monza.csv').
    out_name      : str, filename for the lookup table (default 'sfov_lookup.csv').

    Returns
    -------
    full_path     : str, absolute path of the written CSV file.
    """
    # build DataFrame
    df_out = pd.DataFrame({
        "s_m"   : s_samples,
        "sfov_m": sfov_samples
    })

    # decide output directory = same folder as input file
    out_dir = os.path.dirname(os.path.abspath(in_csv_path))
    full_path = os.path.join(out_dir, out_name)

    # write CSV (no index column)
    df_out.to_csv(full_path, index=False)
    print(f"[INFO] sfov table saved to: {full_path}")

    return full_path


# === example call ===
_ = save_sfov_csv(s_query, sfov_vals, file_path)

In [ ]:
#v1
# ----------------------------------------------------------------------
# ============       PARAMETERS (easy to tune)        ==================
# ----------------------------------------------------------------------
ANG_MIN_DEG   = 7.0          # θ candidates below this are ignored      (°)
CURV_MIN      = 5e-4         # |κ| below this is considered "straight" (1/m)
SLIDING_WIN   = 20.0         # sliding-max window length along s (m)
DENSE_STEP    = 0.5          # dense sampling step for boundaries (m)
DELTA_S       = 300.0        # forward window for FOV (m)

# ----------------------------------------------------------------------
# ============  Helper functions: boundary, extrema, FOV  ==============
# ----------------------------------------------------------------------
from typing import Tuple

def build_boundaries(spline_x, spline_y,
                     w_left: np.ndarray, w_right: np.ndarray,
                     s_dense: np.ndarray
                     ) -> Tuple[
                         Tuple[np.ndarray, np.ndarray],   # left (x,y)
                         Tuple[np.ndarray, np.ndarray],   # right (x,y)
                         Tuple[np.ndarray, np.ndarray],   # normal (nx,ny)
                         np.ndarray                       # tangent norm
                     ]:
    """Return dense left/right boundary samples and local normals."""
    xc = spline_x(s_dense)
    yc = spline_y(s_dense)

    dx = spline_x.derivative(1)(s_dense)
    dy = spline_y.derivative(1)(s_dense)
    norm_t = np.hypot(dx, dy)
    tx, ty = dx / norm_t, dy / norm_t
    nx, ny = -ty, tx                                    # left normal

    w_left_d  = np.interp(s_dense, s, w_left)
    w_right_d = np.interp(s_dense, s, w_right)

    xl = xc + nx * w_left_d
    yl = yc + ny * w_left_d
    xr = xc - nx * w_right_d
    yr = yc - ny * w_right_d
    return (xl, yl), (xr, yr), (nx, ny), norm_t


def first_crossing(s0: float, p_ray: np.ndarray,
                   p0_pos: np.ndarray,          # Vehicle position (x,y)
                   line_x: np.ndarray,        # X-coords of the line to check against
                   line_y: np.ndarray,        # Y-coords of the line to check against
                   s_dense: np.ndarray) -> float:
    i0 = np.searchsorted(s_dense, s0)
    # P0 = np.array([center_x[i0], center_y[i0]]) # This was the old way
    P0 = p0_pos # Use the precise vehicle position

    for i in range(i0 + 1, len(line_x) - 1):
        A = np.array([line_x[i],     line_y[i]])
        B = np.array([line_x[i + 1], line_y[i + 1]])
        v = B - A
        # Ray from P0 in direction p_ray intersects segment AB
        M = np.column_stack((p_ray, -v))
        rhs = A - P0
        try:
            t, u = np.linalg.solve(M, rhs)
        except np.linalg.LinAlgError:
            continue
        
        # t is distance along the ray, must be positive
        # u is fraction along the segment, must be in [0, 1]
        if t > 1e-3 and 0.0 <= u <= 1.0:
            # Return the interpolated arc length 's' of the intersection point
            return s_dense[i] + u * (s_dense[i + 1] - s_dense[i])
            
    return np.inf # No intersection found in the forward window


def has_local_extreme(theta: np.ndarray) -> bool:
    """
    Checks if the sequence of angles is monotonic.
    A change in trend (e.g., from increasing to decreasing) indicates a local extremum.
    """
    if len(theta) < 3:
        return False  # Not enough points to determine a trend change

    # Calculate differences between consecutive elements
    diffs = np.diff(theta)
    
    # Check for a sign change in the differences, ignoring zero-crossings
    # e.g., [..., 0.1, 0.05, -0.02, ...] -> trend changed
    signs = np.sign(diffs[np.abs(diffs) > 1e-6]) # Use a small tolerance for floating point
    if len(signs) < 2:
        return False
        
    return np.any(np.diff(signs) != 0)


def compute_fov_all(spline_x, spline_y,
                    w_left: np.ndarray, w_right: np.ndarray,
                    s_query: np.ndarray,
                    delta_s: float = DELTA_S,
                    step: float = DENSE_STEP) -> np.ndarray:
    """Compute sfov(s) for every point in s_query."""
    # --- dense grid along the whole track --------------------------------
    s_dense = np.arange(0, s_max + step, step)
    (xl, yl), (xr, yr), (nx, ny), _ = build_boundaries(
        spline_x, spline_y, w_left, w_right, s_dense)
    center_x = spline_x(s_dense)
    center_y = spline_y(s_dense)

    sfov_raw = np.empty_like(s_query)

    for k, s0 in enumerate(s_query):
        idx0 = np.searchsorted(s_dense, s0)
        idx_w = np.searchsorted(s_dense, s0 + delta_s, side='right')

        # wrap-around for circular track
        if idx_w >= len(s_dense):
            extra = idx_w - len(s_dense) + 1
            s_ext       = np.concatenate((s_dense, s_dense[1:extra] + s_max))
            xl_ext      = np.concatenate((xl, xl[1:extra]))
            yl_ext      = np.concatenate((yl, yl[1:extra]))
            xr_ext      = np.concatenate((xr, xr[1:extra]))
            yr_ext      = np.concatenate((yr, yr[1:extra]))
            center_xext = np.concatenate((center_x, center_x[1:extra]))
            center_yext = np.concatenate((center_y, center_y[1:extra]))
        else:
            s_ext       = s_dense
            xl_ext, yl_ext = xl, yl
            xr_ext, yr_ext = xr, yr
            center_xext, center_yext = center_x, center_y

        # --- vehicle heading vector at s0 --------------------------------
        tx0 = spline_x.derivative(1)(s0)
        ty0 = spline_y.derivative(1)(s0)
        dir_vec = np.array([tx0, ty0]) / np.hypot(tx0, ty0)

        # --- helper to compute θ sequence --------------------------------
        # --- helper to compute θ sequence (inside compute_fov_all) ----------------
        def theta_arr(xb, yb, s0_pos):
            # s0_pos is the (x,y) coordinate array at s0
            Vx = xb[idx0:idx_w] - s0_pos[0]
            Vy = yb[idx0:idx_w] - s0_pos[1]
            cross = dir_vec[0] * Vy - dir_vec[1] * Vx
            dot   = dir_vec[0] * Vx + dir_vec[1] * Vy
            theta = np.arctan2(cross, dot)
            # NO FILTERING HERE! Let the extremum logic work on raw data.
            return theta, Vx, Vy

        θL, VLx, VLy = theta_arr(xl_ext, yl_ext)
        θR, VRx, VRy = theta_arr(xr_ext, yr_ext)

        has_ext_L = has_local_extreme(θL)
        has_ext_R = has_local_extreme(θR)

        # --- curvature-based straight-road filter ------------------------
        kappa_now = (spline_x.derivative(1)(s0) * spline_y.derivative(2)(s0)
                     - spline_y.derivative(1)(s0) * spline_x.derivative(2)(s0))
        if abs(kappa_now) < CURV_MIN:
            # small curvature: treat as straight
            sfov_raw[k] = s0 + delta_s
            continue

        # if both sides miss extrema within δs, also treat as straight
        if not has_ext_L and not has_ext_R:
            sfov_raw[k] = s0 + delta_s
            continue

        # --- choose rays --------------------------------------------------
        s_inner = np.inf
        s_outer = np.inf

        if kappa_now < 0:  # right turn ⇒ right side is inner
            if has_ext_R:
                idx_R = np.argmin(θR)            # most negative
                p_ray = np.array([VRx[idx_R], VRy[idx_R]])
                p_ray /= np.linalg.norm(p_ray)
                s_inner = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
            if has_ext_L:
                idx_L = np.argmax(θL)            # largest positive
                p_ray = np.array([VLx[idx_L], VLy[idx_L]])
                p_ray /= np.linalg.norm(p_ray)
                s_outer = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
        else:            # left turn
            if has_ext_L:
                idx_L = np.argmax(θL)
                p_ray = np.array([VLx[idx_L], VLy[idx_L]])
                p_ray /= np.linalg.norm(p_ray)
                s_inner = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)
            if has_ext_R:
                idx_R = np.argmin(θR)
                p_ray = np.array([VRx[idx_R], VRy[idx_R]])
                p_ray /= np.linalg.norm(p_ray)
                s_outer = first_crossing(s0, p_ray, center_xext, center_yext, s_ext)

        sfov_raw[k] = min(s_inner, s_outer)

    # ------------------------------------------------------------------
    # Sliding-maximum filter: ensure sfov is non-decreasing
    # ------------------------------------------------------------------
    win = int(SLIDING_WIN / (s_query[1] - s_query[0]))  # window size in samples
    sfov_filt = np.copy(sfov_raw)


    
    # A simple way using pandas for correctness and clarity
    sfov_series = pd.Series(sfov_raw)
    win = int(SLIDING_WIN / (s_query[1] - s_query[0]))
    sfov_filt = sfov_series.rolling(window=win, min_periods=1, center=False).max().to_numpy()

    # Or a manual implementation if you want to avoid pandas here:
    # sfov_filt = np.copy(sfov_raw)
    # for i in range(len(sfov_raw)):
    #     start_idx = max(0, i - win + 1)
    #     sfov_filt[i] = np.max(sfov_raw[start_idx : i + 1])
    return sfov_filt

In [ ]:
# ---------- sfov_interpolant.py ----------
from typing import Union, Callable
import numpy as np
from scipy.interpolate import CubicSpline

def build_sfov_interpolant(s_nodes: np.ndarray,
                           sfov_nodes: np.ndarray,
                           extrapolate: bool = True
                           ) -> Callable[[Union[float, np.ndarray]], np.ndarray]:
    """
    Create a C²-continuous cubic-spline interpolant sfov(s).

    Parameters
    ----------
    s_nodes     : 1-D ndarray, strictly increasing arc-length samples.
    sfov_nodes  : 1-D ndarray, same length, sfov at each sample.
    extrapolate : bool, default True, allow evaluation outside domain.

    Returns
    -------
    sfov_fun    : callable, use like  sfov_fun(s_eval)  →  sfov value(s).
    """
    # Natural BC: second derivative = 0 at both ends
    cs = CubicSpline(s_nodes, sfov_nodes,
                     bc_type="natural",
                     extrapolate=extrapolate)

    def sfov_fun(s_eval: Union[float, np.ndarray]) -> np.ndarray:
        """Vectorised wrapper around CubicSpline."""
        return cs(s_eval)

    return sfov_fun


# === example call & usage ===
sfov_fun = build_sfov_interpolant(s_query, sfov_vals)
print("sfov(123.4 m) =", float(sfov_fun(123.4)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# --------------------------------------------------------------
# sanity-check that required objects are in scope
for _name in ["x_spline", "y_spline", "sfov_fun", "s_max"]:
    if _name not in globals():
        raise RuntimeError(f"Define {_name} before running the animation cell")

# 1. pre-compute background
s_bg = np.linspace(0.0, s_max, 1200)
x_bg = x_spline(s_bg)
y_bg = y_spline(s_bg)

# 2. choose sample frames
n_frames = 400                       # 400 frames ≈ 20 s at 20 fps
s_frames = np.linspace(0.0, s_max, n_frames, endpoint=False)

car_xy  = np.column_stack((x_spline(s_frames),          y_spline(s_frames)))
fov_s   = sfov_fun(s_frames)
fov_xy  = np.column_stack((x_spline(fov_s),             y_spline(fov_s)))
look_ahead = fov_s - s_frames

# 3. build static figure
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(x_bg, y_bg, lw=0.8, color='k', label='Centreline')
car_dot, = ax.plot([], [], 'ro', ms=6, label='Car')
fov_dot, = ax.plot([], [], 'bo', ms=6, label='FOV point')
look_line, = ax.plot([], [], 'b--', lw=1.0)
txt = ax.text(0.02, 0.95, '', transform=ax.transAxes,
              fontsize=9, va='top')
ax.set_aspect('equal')
ax.set_title("Field-of-View Demonstration")
ax.legend(loc='upper right')
ax.set_xlabel('x  [m]')
ax.set_ylabel('y  [m]')
ax.grid(True)

def init():
    car_dot.set_data([], [])
    fov_dot.set_data([], [])
    look_line.set_data([], [])
    txt.set_text('')
    return car_dot, fov_dot, look_line, txt

def update(i):
    cx, cy = car_xy[i]
    fx, fy = fov_xy[i]
    car_dot.set_data(cx, cy)
    fov_dot.set_data(fx, fy)
    look_line.set_data([cx, fx], [cy, fy])
    txt.set_text(f's = {s_frames[i]:.1f} m\n'
                 f'look-ahead = {look_ahead[i]:.1f} m')
    return car_dot, fov_dot, look_line, txt

anim = FuncAnimation(fig, update, frames=n_frames,
                     init_func=init, blit=True, interval=50)

# 4. save as GIF and display
gif_path = "fov_demo.gif"
anim.save(gif_path, writer=PillowWriter(fps=20))
plt.close(fig)        # prevent duplicate static image

display(Image(filename=gif_path))

In [ ]:
dbg = sfov_vals - s_query
idx = np.where(dbg < 2.0)[0]          # 找出 look-ahead < 2 m 的样本
print("suspect s =", s_query[idx])

In [ ]:
# ----------------------------------------------
# Visualise suspect look-ahead samples (< 2 m)
# ----------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

span = 40

# --------- 1. 找出异常样本 ---------------------
look_ahead = sfov_vals - s_query
suspect_idx = np.where(look_ahead < 2.0)[0]      # <—— 阈值可调
print("suspect s:", s_query[suspect_idx])

# 只挑前两个来画；你也可以手动指定 idx_list
idx_list = suspect_idx[:2]

# --------- 2. 预备背景（中心线 & 可选边界） -----
s_bg  = np.linspace(0, s_max, 1200)
x_bg  = x_spline(s_bg)
y_bg  = y_spline(s_bg)

# 若想同时画左右边界，取消下方注释
# w_left_bg  = np.interp(s_bg, s, w_left)
# w_right_bg = np.interp(s_bg, s, w_right)
# dx_bg = x_spline.derivative(1)(s_bg); dy_bg = y_spline.derivative(1)(s_bg)
# norm = np.hypot(dx_bg, dy_bg)
# nx_bg, ny_bg = -dy_bg / norm, dx_bg / norm
# xl_bg = x_bg + nx_bg * w_left_bg;  yl_bg = y_bg + ny_bg * w_left_bg
# xr_bg = x_bg - nx_bg * w_right_bg; yr_bg = y_bg - ny_bg * w_right_bg

for idx in idx_list:
    s0     = s_query[idx]
    s_fov  = sfov_vals[idx]

    # 车辆位置 / FOV 交点
    car_xy = np.array([x_spline(s0),     y_spline(s0)])
    fov_xy = np.array([x_spline(s_fov),  y_spline(s_fov)])

    # 车辆当前射线方向（只是用两个点连线即可）
    ray_vec = fov_xy - car_xy

    # --------- 3. 画图 -------------------------
    plt.figure(figsize=(8, 10), dpi=200)   # figsize 单位英寸, dpi 越大像素越多

    plt.scatter(*fov_xy, c='b', zorder=5, label=f'FOV  s={s_fov:.2f}')

    # ⬇⬇ 关键：限制坐标轴到车辆 ±span/2 ⬇⬇
    plt.xlim(car_xy[0] - span/2, car_xy[0] + span/2)
    plt.ylim(car_xy[1] - span/2, car_xy[1] + span/2)

    plt.gca().set_aspect('equal')


    plt.plot(x_bg, y_bg, 'k', lw=0.8, label='centreline')

    # 如需边界，取消注释
    # plt.plot(xl_bg, yl_bg, '0.7', lw=0.5)
    # plt.plot(xr_bg, yr_bg, '0.7', lw=0.5)

    plt.plot([car_xy[0], fov_xy[0]], [car_xy[1], fov_xy[1]],
             'r--', lw=1.2, label='ray')
    plt.scatter(*car_xy, c='r', zorder=5, label=f'car  s={s0:.2f}')
    plt.scatter(*fov_xy, c='b', zorder=5, label=f'FOV  s={s_fov:.2f}')

    plt.gca().set_aspect('equal')
    plt.title(f'Look-ahead = {s_fov - s0:.2f} m')
    plt.legend()
    plt.grid(True)
    plt.show()